In [34]:
import pandas as pd
from hmm_logic_methods import load_model
import pomegranate
from pomegranate.distributions import DiscreteDistributionCycle
import os
EXPERIMENT_NAME = 'multiple_single_allele_models'

In [35]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
DATA_PATH = f'{DATA_PATH_BASE}/{EXPERIMENT_NAME}/' #Work laptop

In [36]:
class_to_use = 'II'
organism = 'Human'
DECOY_DATA_PATH = f"{DATA_PATH_BASE}\decoy\sampled_decoy_data_{organism}_class_{class_to_use}.csv"
decoy_df =  pd.read_csv(DECOY_DATA_PATH)
decoy_df = decoy_df.sample(frac=1).reset_index(drop=True)
decoy_df =decoy_df.iloc[:int(len(decoy_df)/12000)*12000]
decoy_df['core'] = decoy_df.peptide.str.slice(0,9)
decoy_df['core_len'] = decoy_df.core.str.len()
decoy_df['allele'] = [ f'decoy-allele-{j}'  for j in range(int(len(decoy_df)/12000)) for i in range(12000)]
decoy_df['binder'] = 1
len(decoy_df)

3396000

## Make per allele df

In [37]:
%%time
ALLELES = list( decoy_df.allele.unique())
per_allele_train_df = {allele_name: decoy_df[decoy_df.allele == allele_name].copy() for allele_name in ALLELES}


CPU times: total: 37.1 s
Wall time: 37.5 s


# 2. Make per length split

In [38]:
from sklearn.model_selection import KFold
MIN_LENGTH = 8
MAX_LENGTH = 30

#leave only binders
def getBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 1]['peptide'].values
    return result_peptides
def getNonBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 0]['peptide'].values
    return result_peptides

def make_per_length_dict(df, binder_flag):
    lengths = set(df['length'])
    lengths = set([length for length in lengths if length >= MIN_LENGTH and length <= MAX_LENGTH])
    per_length_dict = dict()
    for length in lengths:
        result_peptides = df[(df['binder'] == binder_flag) & (df['length'] == length)]['peptide'].values
        per_length_dict[length] = result_peptides
    return per_length_dict

kf = KFold(n_splits=5, shuffle=True)
def make_per_length_per_KFold_dict(per_length_dict):

    per_length_per_kfold_peptides_train = dict()
    per_length_per_kfold_peptides_test = dict()

    print("Length \t Train/Test for splits", end="")
    for target_length in per_length_dict.keys():
        print(f"\n{target_length}:", end="\t")
        peptides_array = per_length_dict[target_length]
        per_length_per_kfold_peptides_train[target_length] = dict()
        per_length_per_kfold_peptides_test[target_length] = dict()
        if len(peptides_array) > kf.n_splits:
            i = 0
            for train_indexes_b, test_indexes_b in kf.split(peptides_array):
                print("{}/{}\t".format(len(train_indexes_b), len(test_indexes_b)), end="")
                current_train = peptides_array[train_indexes_b]
                current_test = peptides_array[test_indexes_b]
                per_length_per_kfold_peptides_train[target_length][i] = current_train
                per_length_per_kfold_peptides_test[target_length][i] = current_test
                i = i + 1
        else:

            print(f"Not enough peptides of length {target_length}", end="")
    print()
    return per_length_per_kfold_peptides_train , per_length_per_kfold_peptides_test





def addAuxColumns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        df['length'] = df['peptide'].str.len()

per_allele_binders_train = dict()
per_allele_non_binders_train = dict()
per_allele_per_length_binders_train = dict()
per_allele_per_length_non_binders_train = dict()
per_allele_per_length_per_kfold_binders_train = dict()
per_allele_per_length_per_kfold_binders_test = dict()
per_allele_per_length_per_kfold_non_binders_train = dict()
per_allele_per_length_per_kfold_non_binders_test = dict()


for allele_name in ALLELES:
    print(f"Allele {allele_name}")
    df_train = per_allele_train_df[allele_name]
    df_train = df_train.sample(frac=1).reset_index(drop=True)
    per_allele_binders_train[allele_name] = getBindersPeptidesOnly(df_train)
    per_allele_non_binders_train[allele_name] = getNonBindersPeptidesOnly(df_train)
    per_allele_per_length_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=1)
    per_allele_per_length_non_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=0)
    print(f"Binders")
    per_allele_per_length_per_kfold_binders_train[allele_name],  per_allele_per_length_per_kfold_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_binders_train[allele_name])
    print(f"Non-binders")
    per_allele_per_length_per_kfold_non_binders_train[allele_name],  per_allele_per_length_per_kfold_non_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_non_binders_train[allele_name])


Allele decoy-allele-0
Binders
Length 	 Train/Test for splits
11:	564/142	565/141	565/141	565/141	565/141	
12:	542/136	542/136	542/136	543/135	543/135	
13:	548/138	549/137	549/137	549/137	549/137	
14:	572/143	572/143	572/143	572/143	572/143	
15:	548/138	549/137	549/137	549/137	549/137	
16:	579/145	579/145	579/145	579/145	580/144	
17:	580/145	580/145	580/145	580/145	580/145	
18:	552/138	552/138	552/138	552/138	552/138	
19:	564/141	564/141	564/141	564/141	564/141	
20:	546/137	546/137	546/137	547/136	547/136	
21:	549/138	549/138	550/137	550/137	550/137	
22:	558/140	558/140	558/140	559/139	559/139	
23:	561/141	561/141	562/140	562/140	562/140	
24:	596/150	597/149	597/149	597/149	597/149	
25:	576/144	576/144	576/144	576/144	576/144	
26:	576/144	576/144	576/144	576/144	576/144	
27:	583/146	583/146	583/146	583/146	584/145	
Non-binders
Length 	 Train/Test for splits
11:	Not enough peptides of length 11
12:	Not enough peptides of length 12
13:	Not enough peptides of length 13
14:	Not enough pepti

# 3. Save per_length_per_split to files

In [39]:
import os
from collections import defaultdict

TARGET_PATH_TO_FILES = DATA_PATH  + '/per_length_per_kfold_split/'
if not os.path.exists(TARGET_PATH_TO_FILES):
    os.makedirs(TARGET_PATH_TO_FILES)

def save_data(per_allele_per_length_per_kfold_data, data_type=None, dataset_type=None):
    assert data_type in ["binders", "nonbinders"]
    assert dataset_type in ["train", "test"]

    per_allele_dfs = defaultdict(list)
    for allele_name in per_allele_per_length_per_kfold_data.keys():
        for length, per_k_fold_dict in per_allele_per_length_per_kfold_data[allele_name].items():
                for split_num, peptides in per_k_fold_dict.items():
                    df = pd.DataFrame({"peptide":peptides, "length":length, "split": split_num})
                    per_allele_dfs[allele_name].append(df)
        total_df= pd.concat(per_allele_dfs[allele_name])
        target_allele_name = allele_name.replace("*", "_")
        total_df.to_csv(f'{TARGET_PATH_TO_FILES}{target_allele_name}#{data_type}#{dataset_type}.csv',sep=";", index=False)


save_data(per_allele_per_length_per_kfold_binders_train, "binders", "train")
# save_data(per_allele_per_length_per_kfold_non_binders_train, "nonbinders", "train")
save_data(per_allele_per_length_per_kfold_binders_test, "binders", "test")
# save_data(per_allele_per_length_per_kfold_non_binders_test, "nonbinders", "test")

In [28]:
ALLELES

['decoy_allele_0',
 'decoy_allele_1',
 'decoy_allele_2',
 'decoy_allele_3',
 'decoy_allele_4',
 'decoy_allele_5',
 'decoy_allele_6',
 'decoy_allele_7',
 'decoy_allele_8',
 'decoy_allele_9',
 'decoy_allele_10',
 'decoy_allele_11',
 'decoy_allele_12',
 'decoy_allele_13',
 'decoy_allele_14',
 'decoy_allele_15',
 'decoy_allele_16',
 'decoy_allele_17',
 'decoy_allele_18',
 'decoy_allele_19',
 'decoy_allele_20',
 'decoy_allele_21',
 'decoy_allele_22',
 'decoy_allele_23',
 'decoy_allele_24',
 'decoy_allele_25',
 'decoy_allele_26',
 'decoy_allele_27',
 'decoy_allele_28',
 'decoy_allele_29',
 'decoy_allele_30',
 'decoy_allele_31',
 'decoy_allele_32',
 'decoy_allele_33',
 'decoy_allele_34',
 'decoy_allele_35',
 'decoy_allele_36',
 'decoy_allele_37',
 'decoy_allele_38',
 'decoy_allele_39',
 'decoy_allele_40',
 'decoy_allele_41',
 'decoy_allele_42',
 'decoy_allele_43',
 'decoy_allele_44',
 'decoy_allele_45',
 'decoy_allele_46',
 'decoy_allele_47',
 'decoy_allele_48',
 'decoy_allele_49',
 'decoy_al